In [6]:
import pandas as pd
from dotenv import load_dotenv
import os
from datetime import date, timedelta
import time
import json
import ijson
from sqlalchemy import create_engine
from tqdm import tqdm

In [7]:
import sys
sys.path.append('./lib')

import library_st_data_processing as lsdp

In [8]:
# load .env from the current directory (or specify a path)
load_dotenv(dotenv_path=".env")
api_key = os.getenv("ST_API_KEY")

In [9]:
base_url = 'https://api.sensortower.com'

In [10]:
today_str = date.today().strftime("%Y-%m-%d")
timestamp = int(time.time())

# Base Layer

**Get Raw File: Top Game Annual Performance**

In [5]:
# Manual Download

**Get Raw File: Game Full Info**

In [8]:
# Load Top Game Annual Performance file
df_top_game_annual_performance = pd.read_csv("data/base/st_webdownload_annual_game_performance_2025-10-23.csv")

In [10]:
# Get the unified_app_ids
unified_app_ids = set(list(df_top_game_annual_performance["Unified ID"]))

In [18]:
# Retrieve Game Full Info file
df_game_full_info = lsdp.retrieve_full_info_game_table(api_key, base_url, unified_app_ids)

Get canonical app data...
making call for chunk number 1
call successful!
making call for chunk number 2
call successful!
making call for chunk number 3
call successful!
making call for chunk number 4
call successful!
making call for chunk number 5
call successful!
making call for chunk number 6
call successful!
making call for chunk number 7
call successful!
making call for chunk number 8
call successful!
making call for chunk number 9
call successful!
making call for chunk number 10
call successful!
making call for chunk number 11
call successful!
making call for chunk number 12
call successful!
making call for chunk number 13
call successful!
making call for chunk number 14
call successful!
making call for chunk number 15
call successful!
making call for chunk number 16
call successful!
making call for chunk number 17
call successful!
making call for chunk number 18
call successful!
making call for chunk number 19
call successful!
making call for chunk number 20
call successful!
mak

In [19]:
# Export the file to csv and save to base layer folder
df_game_full_info.to_csv('data/base/st_api_game_full_info_{}.csv'.format(timestamp), index=False)

**Get Raw File: App Full Info**

In [6]:
# Read file Game Full Info
df_game_full_info = pd.read_csv('data/base/st_api_game_full_info_2025-10-23.csv')

In [8]:
# Retrieve file App Full Info
df_app_full_info = lsdp.get_local_app_info_from_game_full_info_table(api_key, base_url, df_game_full_info)

Extracting app list from game table...
Getting iOS apps info...
making call for chunk number 1
call successful!
making call for chunk number 2
call successful!
making call for chunk number 3
call successful!
making call for chunk number 4
call successful!
making call for chunk number 5
call successful!
making call for chunk number 6
call successful!
making call for chunk number 7
call successful!
making call for chunk number 8
call successful!
making call for chunk number 9
call successful!
making call for chunk number 10
call successful!
making call for chunk number 11
call successful!
making call for chunk number 12
call successful!
making call for chunk number 13
call successful!
making call for chunk number 14
call successful!
making call for chunk number 15
call successful!
making call for chunk number 16
call successful!
making call for chunk number 17
call successful!
making call for chunk number 18
call successful!
making call for chunk number 19
call successful!
making call fo

In [9]:
# Export the file to csv and save to base layer folder
df_app_full_info.to_csv('data/base/st_api_app_full_info_{}.csv'.format(timestamp), index=False)

**Get Raw File: Mapping Publisher to Apps (Top Down)**

In [7]:
# Manually Prepared

**Get Raw File: Mapping Publisher to Apps and Publisher IDs (Top Down)**

In [20]:
# Path to Mapping Publisher to Apps Raw File
path_mapping_publisher_to_apps = 'data/base/st_manualsearch_mapping_publisher_to_apps_top_down_2025-10-28.csv'

In [21]:
# Retrieve the publisherid from ST
df_mapping_publisher_to_apps_and_publisherids = lsdp.create_mapping_publisher_to_apps_publisherids(path_mapping_publisher_to_apps, api_key, base_url)

Reading csv file: mapping publisher to apps...
Get ST publisher info for ios apps...
making call for chunk number 1
call successful!
making call for chunk number 2
call successful!
making call for chunk number 3
call successful!
making call for chunk number 4
call successful!
Get ST publisher info for Android apps...
making call for chunk number 1
call successful!
making call for chunk number 2
call successful!
making call for chunk number 3
call successful!
making call for chunk number 4
call successful!


In [22]:
# Export the file to csv and save to base layer folder
df_mapping_publisher_to_apps_and_publisherids.to_csv('data/base/st_api_mapping_publisher_to_apps_publisherids_top_down_{}.csv'.format(timestamp), index=False)

**Get Raw File: Mapping Publisher to Publisher ID and Revenue Multiplier (Bottom Up)**

In [12]:
# Manually Prepared

**Get Raw File: Mapping App to Revenue Multiplier (Special Cases)**

In [ ]:
# Manually Prepared

**Get Raw File: App Performance (Daily)**

In [6]:
# Read file App Full Info (Adjusted)
df_app_full_info_adjusted = pd.read_csv('data/staging/st_app_full_info_adjusted_1763107898.csv', low_memory=False)

In [ ]:
# Run the function to fetch the data and export to json
app_performance_data = lsdp.create_table_app_performance_grouped_by_game_daily(
    api_key, 
    base_url, 
    str_start_date = '2014-01-01', 
    str_end_date = (date.today() - timedelta(days=3)).isoformat(), 
    df_app_full_info_adjusted, 
    json_export_path = 'data/base')

# Staging Layer

**Get Staging File: Mapping Publisher to Publisher ID and Revenue Multiplier (Full)**

In [6]:
# Read file Mapping Publisher to Apps and Publisher IDs (Top Down)

data_types = {
    "os_x": "string",
    "cleaned_publisher_name": "string",
    "game_name": "string",
    "sensor_tower_link": "string",
    "app_id_trimmed": "string",
    "publisher_id": "string",
    "publisher_name": "string",
}

df_mapping_publisher_to_apps_and_publisherids_top_down = pd.read_csv(
    'data/base/st_api_mapping_publisher_to_apps_publisherids_top_down_2025-10-28.csv',
    dtype = data_types
)

In [7]:
# Read file Mapping Publisher to Publisher ID and Revenue Multiplier (Bottom Up)

data_types = {
    "cleaned_publisher_name": "string",
    "publisher_id": "string",
    "publisher_name": "string",
    "revenue_multiplier": "int64"
}

df_mapping_publisher_to_publisherid_revenue_multiplier_bottom_up = pd.read_csv(
    'data/base/st_manualsearch_mapping_publisher_to_publisherids_revenue_multiplier_bottom_up_2025-10-31.csv',
    dtype = data_types
)

In [8]:
# Merge the top-down and bottom-up tables into a full mapping

In [9]:
df_mapping_publisher_to_publisherid_revenue_multiplier_full = lsdp.create_mapping_publisher_to_publisherids_revenue_multiplier(
    df_mapping_publisher_to_apps_and_publisherids_top_down,
    df_mapping_publisher_to_publisherid_revenue_multiplier_bottom_up
)

Merging is complete, now have the full mapping table. But need to double check for rows having null publisher_id
These are rows having null publisher_id:
             cleaned_publisher_name publisher_id publisher_name  \
138  CTCP Giai Tri Thien Thuong Hoa         <NA>           <NA>   

     revenue_multiplier  
138                   3  
Best to check again these rows. Potential reasons: apps become inactive in the country of interest; app id typo; app id changed by SensorTower; etc.
Removing these rows from the merged mapping table...


In [10]:
# Export the file to csv and save to staging layer folder
df_mapping_publisher_to_publisherid_revenue_multiplier_full.to_csv("data/staging/st_mapping_publisher_to_publisherids_revenue_multiplier_full_{}.csv".format(timestamp),index=False)

**Get Staging File: App Full Info (Adjusted)**

In [6]:
# Read file App Full Info
df_app_full_info = pd.read_csv('data/base/st_api_app_full_info_2025-10-27.csv', low_memory=False)

In [7]:
# Read file Mapping Publisher to Publisher ID and Revenue Multiplier (Full)
df_mapping_publisher_to_publiserid_and_revenue_multiplier_full = pd.read_csv('data/staging/st_mapping_publisher_to_publisherids_revenue_multiplier_full_2025-10-31.csv', low_memory=False)

In [8]:
# Read file Mapping App to Revenue Multiplier (Special Cases)
df_mapping_app_to_revenue_multiplier_special_case = pd.read_csv('data/base/st_mapping_app_to_revenue_multiplier_special_case_1763095974.csv', low_memory=False)

In [9]:
# Cast cleaned_publisher_name and revenue_multiplier to App Full Info to get App Full Info (Adjusted)
df_app_full_info_adjusted = lsdp.add_cleaned_publisher_name_and_revenue_multiplier_to_app_full_info(
    df_app_full_info,
    df_mapping_publisher_to_publiserid_and_revenue_multiplier_full
)

In [27]:
# Adjust the App Full Info list again for special cases
df_app_full_info_adjusted_again = lsdp.adjust_cleaned_publisher_name_and_revenue_multiplier_of_app_full_info_special_cases(
    df_app_full_info_adjusted,
    df_mapping_app_to_revenue_multiplier_special_case
).drop_duplicates()

In [34]:
# Export the file to csv and save to staging layer folder
df_app_full_info_adjusted_again.to_csv('data/staging/st_app_full_info_adjusted_{}.csv'.format(timestamp), index=False)

**Get Staging File: App Performance (Daily) - Revenue Adjusted [NEED TO EDIT LATER WITH CORRECT FILE PATHS & DO SOME REFACTOR]**

In [22]:
# Read file App Full Info (Adjusted)
df_app_full_info_adjusted = pd.read_csv('data/staging/st_app_full_info_adjusted_1763107898.csv', low_memory=False)

In [14]:
# Specify the path of input and output
src_path = "data/staging/st_app_performance_daily_1764146538.json"
file_timestamp = os.path.basename(src_path).split("_")[-1].split(".")[0]
dst_path = "data/staging/st_app_performance_daily_{}_revenue_adjusted.json".format(file_timestamp)

In [ ]:
# Run the function for adjusting and streaming
lsdp.stream_and_adjust_app_performance_daily_json_file(
    src_path,
    dst_path,
    df_app_full_info_adjusted
)

In [41]:
# Get the mapping app_id => revenue_multiplier
df_mult = df_app_full_info_adjusted[df_app_full_info_adjusted['revenue_multiplier']!=1][['app_id','revenue_multiplier']].copy()
revenue_factor_by_app = (
    df_mult.set_index("app_id")["revenue_multiplier"].to_dict()
)

In [46]:
def adjust_revenue(rec):
    aid_str = str(rec.get("aid"))
    factor = revenue_factor_by_app.get(aid_str)

    # If no multiplier (or factor == 1), just return as-is
    if factor is None or factor == 1:
        return rec

    # iOS: ir = iPhone revenue, ar = iPad revenue
    if "ir" in rec and rec["ir"] is not None:
        rec["ir"] = rec["ir"] * factor
    if "ar" in rec and rec["ar"] is not None:
        rec["ar"] = rec["ar"] * factor

    # Android: r = Android revenue
    if "r" in rec and rec["r"] is not None:
        rec["r"] = rec["r"] * factor

    return rec

In [49]:
# Streaming and adjusting original app performance data file to adjusted app performance data file
src_path = "data/staging/st_app_performance_daily_1764146538.json"
dst_path = "data/staging/st_app_performance_daily_1764146538_revenue_adjusted.json"

total_bytes = os.path.getsize(src_path)

with open(src_path, "rb") as src, \
     open(dst_path, "w", encoding="utf-8") as dst, \
     tqdm(total=total_bytes, unit="B", unit_scale=True, desc="Processing") as pbar:

    dst.write("[\n")
    first = True

    for rec in ijson.items(src, "item"):
        rec = adjust_revenue(rec)

        if not first:
            dst.write(",\n")
        first = False

        json.dump(rec, dst, ensure_ascii=False)

        # update progress to current file position
        pbar.update(src.tell() - pbar.n)

    dst.write("\n]\n")


rocessing: 100%|█████████████████████████████████████████████████████████████████| 7.10G/7.10G [22:15<00:00, 5.32MB/s]

In [51]:
# Checking: read the output json file
def count_items_with_progress(path):
    total_bytes = os.path.getsize(path)
    count = 0

    with open(path, "rb") as f, \
         tqdm(
             total=total_bytes,
             unit="B",
             unit_scale=True,
             desc=f"Counting {os.path.basename(path)}"
         ) as pbar:

        # 'item' iterates each element of the top-level JSON array
        for _ in ijson.items(f, "item"):
            count += 1
            # advance progress to current file position
            pbar.update(f.tell() - pbar.n)

    print(f"{path} → {count:,} records")
    return count


count_items_with_progress("data/staging/st_app_performance_daily_1764146538.json")
count_items_with_progress("data/staging/st_app_performance_daily_1764146538_revenue_adjusted.json")


ounting st_app_performance_daily_1764146538.json: 100%|██████████████████████████| 7.10G/7.10G [04:24<00:00, 26.9MB/s]

data/staging/st_app_performance_daily_1764146538.json → 52,747,279 records


Counting st_app_performance_daily_1764146538_revenue_adjusted.json: 100%|█████████| 4.45G/4.45G [04:11<00:00, 17.7MB/s]

data/staging/st_app_performance_daily_1764146538_revenue_adjusted.json → 52,747,279 records


52747279

In [42]:
revenue_factor_by_app

{'6476554439': 3,
 '904890205': 20,
 '1521771561': 3,
 '1621483610': 3,
 '6443442131': 3,
 '6740561197': 3,
 '6740611773': 3,
 '6747331870': 3,
 '1431610213': 3,
 '6504474986': 3,
 '6451137140': 3,
 '1127157068': 15,
 '1159236542': 3,
 '1261834568': 3,
 '1577123983': 3,
 '1438376417': 3,
 '1469164964': 3,
 '6466761725': 15,
 '6479692189': 3,
 '1583717977': 3,
 '6747154050': 3,
 '6444060858': 3,
 '1591546813': 3,
 '1588009371': 15,
 '6448143057': 3,
 '6717587690': 3,
 '6740536674': 15,
 '799319182': 20,
 '887752917': 20,
 '1528287521': 15,
 '6633416904': 15,
 '1573033171': 3,
 '1495625837': 3,
 '6446198142': 3,
 '1616630100': 3,
 '6499431484': 3,
 '1665137451': 3,
 '6478189835': 3,
 '6479636678': 3,
 '6476176288': 3,
 '1521411821': 3,
 '6467690444': 3,
 '6478138913': 3,
 '638689075': 3,
 '6480434384': 3,
 '6446388135': 3,
 '6451463643': 15,
 '1546606867': 15,
 '6738641269': 3,
 '1460786661': 3,
 '1633945760': 3,
 '6478901170': 15,
 '6504267531': 15,
 '1071744151': 3,
 '1593143999': 3,
 

In [43]:
factor = revenue_factor_by_app.get("abx")

In [45]:
factor == None

True

In [28]:
df_app_full_info_adjusted

,app_id,canonical_country,name,publisher_name,publisher_id,humanized_name,icon_url,os,active,url,...,screenshot_urls,tablet_screenshot_urls,description,subtitle,promo_text,permissions,supported_languages,country_release_date,cleaned_publisher_name,revenue_multiplier
0,336834650,US,Moorhuhn Deluxe!,Acmee GmbH,330311834,Moorhuhn Deluxe!,https://is1-ssl.mzstatic.com/image/thumb/Purpl...,ios,False,https://apps.apple.com/US/app/id336834650?l=en,...,['https://is3-ssl.mzstatic.com/image/thumb/Pur...,['https://is3-ssl.mzstatic.com/image/thumb/Pur...,"In this hunting game for iPhone, iPad and iPod...",Crazy Chicken,NaN,NaN,"['en', 'fr', 'de', 'it', 'es']",2009-11-03T08:00:00Z,NaN,1
1,373942073,US,Real Solitaire for iPad,"EdgeRift, Inc.",292786128,Real Solitaire,https://is1-ssl.mzstatic.com/image/thumb/Purpl...,ios,True,https://apps.apple.com/US/app/id373942073?l=en,...,[],[],Play the best new free Solitaire! Millions of ...,Classic Solitaire puzzle game.,NaN,NaN,['en'],2010-06-18T00:00:00Z,NaN,1
2,457859877,US,Kick it out! Football Manager,Ludetis UG,457859880,Kick it out! Football Manager,https://is2-ssl.mzstatic.com/image/thumb/Purpl...,ios,False,https://apps.apple.com/US/app/id457859877?l=en,...,['https://is1-ssl.mzstatic.com/image/thumb/Pur...,[],Football is emotion - and Kick it out! is the ...,NaN,NaN,NaN,"['bg', 'nl', 'en', 'fr', 'de', 'el', 'it', 'pl...",2012-05-10T00:39:29Z,NaN,1
3,472626879,US,Coin Hunter 2,Easeware,448134626,Coin Hunter 2,https://is2-ssl.mzstatic.com/image/thumb/Purpl...,ios,True,https://apps.apple.com/US/app/id472626879?l=en,...,['https://is1-ssl.mzstatic.com/image/thumb/Pur...,[],"By the leads of the map father left, the brave...",NaN,NaN,NaN,['en'],2011-10-19T05:14:22Z,NaN,1
4,472885640,US,JJ斗地主-欢乐棋牌休闲合集,JJWorld(Beijing) Network Technology Company Li...,472885643,JJ斗地主-欢乐棋牌休闲合集,https://is1-ssl.mzstatic.com/image/thumb/Purpl...,ios,True,https://apps.apple.com/US/app/id472885640?l=en,...,['https://is1-ssl.mzstatic.com/image/thumb/Pur...,[],JJ斗地主牌局专业，赛制公平，无猪队友，无作弊，斗得过瘾，赢得畅快。24小时比赛不间断，回馈...,数亿玩家的口碑之选,NaN,NaN,[],2012-02-29T00:00:00Z,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80475,onetap.game.lines98.codien,US,Lines 98 Cổ Điển,ONETAP,ONETAP,Lines 98 Cổ Điển,https://play-lh.googleusercontent.com/whs8Io-c...,android,False,https://play.google.com/store/apps/details?hl=...,...,['https://play-lh.googleusercontent.com/r15UVV...,[],Lines 98 cổ điển dành cho những ai yêu thích t...,NaN,NaN,{'Wi-Fi connection information': ['view Wi-Fi ...,[],2022-05-12T00:00:00Z,NaN,1
80476,se.illusionlabs.bmx,US,Touchgrind BMX,Illusion Labs,Illusion+Labs,Touchgrind BMX,https://play-lh.googleusercontent.com/OphBEeHx...,android,True,https://play.google.com/store/apps/details?hl=...,...,['https://play-lh.googleusercontent.com/YgNp3k...,[],Become a BMX pro and perform spectacular trick...,NaN,NaN,"{'Other': ['view network connections', 'full n...",[],2014-09-12T00:00:00Z,NaN,1
80477,sk.sivak.eldritchhorror.android,US,Ancient Terror: Lovecraftian S,Pixel Art Battles,Pixel+Art+Battles,Ancient Terror,https://play-lh.googleusercontent.com/v2d5kJN8...,android,False,https://play.google.com/store/apps/details?hl=...,...,['https://play-lh.googleusercontent.com/oSqasN...,[],PC version: https://ancientterror.itch.io/anci...,NaN,NaN,{'Photos/Media/Files': ['read the contents of ...,[],2019-03-29T00:00:00Z,NaN,1
80478,word.hunt.connect,US,Word Hunt: Word Puzzle Game,Word Search Games,Word+Search+Games,Word Hunt: Word Puzzle Game,https://play-lh.googleusercontent.com/je-v1alA...,android,True,https://play.google.com/store/apps/details?hl=...,...,['https://play-lh.googleusercontent.com/P27Txm...,[],"Play Word Hunt 9 mins a day, search the words,...",NaN,NaN,{'Wi-Fi connection information': ['view Wi-Fi ...,[],2019-08-09T00:00:00Z,NaN,1


In [31]:
# Get the list of Apps requiring revenue adjustment and their multipliers
df_app_and_revenue_multiplier = df_app_full_info_adjusted[df_app_full_info_adjusted['revenue_multiplier']!=1][['app_id','revenue_multiplier']]

In [33]:
for app_id in df_app_and_revenue_multiplier['app_id']:
    print(app_id)

6476554439
904890205
1521771561
1621483610
6443442131
6740561197
6740611773
6747331870
1431610213
6504474986
6451137140
1127157068
1159236542
1261834568
1577123983
1438376417
1469164964
6466761725
6479692189
1583717977
6747154050
6444060858
1591546813
1588009371
6448143057
6717587690
6740536674
799319182
887752917
1528287521
6633416904
1573033171
1495625837
6446198142
1616630100
6499431484
1665137451
6478189835
6479636678
6476176288
1521411821
6467690444
6478138913
638689075
6480434384
6446388135
6451463643
1546606867
6738641269
1460786661
1633945760
6478901170
6504267531
1071744151
1593143999
1502412815
1467842750
1521779642
1582020814
6479231049
6446610044
1417141579
1444754600
1464689103
1587704770
6737103277
1559698151
1465097065
6453690496
1306451576
6471930550
1453657959
6472704324
6469768541
1501476139
1635925350
1477780217
6449825814
1632354684
6448864292
6502928253
1489436528
1546353393
6456407206
6502480712
6596734272
6480383911
6443444765
1451281983
1570594577
1472743924
673

In [39]:
df_app_and_revenue_multiplier[df_app_and_revenue_multiplier['app_id']=='1573033171']['revenue_multiplier'].item()

3

In [ ]:
for app_id in df_app_and_revenue_multiplier['app_id']:
    print("Adjusting for aid: {}".format(app_id))
    for record in 

In [40]:
app_performance_data

[{'aid': 336834650, 'cc': 'VN', 'd': '2014-01-13T00:00:00Z', 'ir': 8},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-17T00:00:00Z', 'ir': 4},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-19T00:00:00Z', 'ir': 6},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-22T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-28T00:00:00Z', 'ir': 1},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-29T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-02-13T00:00:00Z', 'ir': 2},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-03-01T00:00:00Z', 'ir': 1},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-03-12T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-13T00:00:00Z', 'ir': 2},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-19T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-22T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-27T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-07-04T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-07-06T00:00:00Z', 'ir': 4}

# Modeling Layer

In [9]:
# read data from json
with open('data/draft/app_performance_data_1764146538.json', 'r') as file:
    app_performance_data = json.load(file)

In [20]:
app_performance_data

[{'aid': 336834650, 'cc': 'VN', 'd': '2014-01-13T00:00:00Z', 'ir': 8},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-17T00:00:00Z', 'ir': 4},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-19T00:00:00Z', 'ir': 6},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-22T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-28T00:00:00Z', 'ir': 1},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-29T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-02-13T00:00:00Z', 'ir': 2},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-03-01T00:00:00Z', 'ir': 1},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-03-12T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-13T00:00:00Z', 'ir': 2},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-19T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-22T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-27T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-07-04T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-07-06T00:00:00Z', 'ir': 4}

In [26]:
# group the data by app id and unified id

df_perf = pd.DataFrame(app_performance_data)

In [29]:
df_perf

,aid,cc,d,ir,au,iu,ar,c,u,r,aid_str
0,336834650,VN,2014-01-13T00:00:00Z,8,NaN,NaN,NaN,NaN,NaN,NaN,336834650
1,336834650,VN,2014-01-17T00:00:00Z,4,NaN,NaN,NaN,NaN,NaN,NaN,336834650
2,336834650,VN,2014-01-19T00:00:00Z,6,NaN,NaN,NaN,NaN,NaN,NaN,336834650
3,336834650,VN,2014-01-22T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,336834650
4,336834650,VN,2014-01-28T00:00:00Z,1,NaN,NaN,NaN,NaN,NaN,NaN,336834650
...,...,...,...,...,...,...,...,...,...,...,...
52747274,world.playme.x,NaN,2025-10-20T00:00:00Z,NaN,NaN,NaN,NaN,VN,NaN,68,world.playme.x
52747275,world.playme.x,NaN,2025-10-21T00:00:00Z,NaN,NaN,NaN,NaN,VN,NaN,81,world.playme.x
52747276,world.playme.x,NaN,2025-10-22T00:00:00Z,NaN,NaN,NaN,NaN,VN,NaN,NaN,world.playme.x
52747277,world.playme.x,NaN,2025-10-23T00:00:00Z,NaN,NaN,NaN,NaN,VN,NaN,NaN,world.playme.x


In [28]:
# Make sure app ids are consistent types (strings in this example)
df_perf["aid_str"] = df_perf["aid"].astype(str)

In [30]:
df_perf["d"] = pd.to_datetime(df_perf["d"])

In [31]:
# Make a string version of app_id too
df_app_full_info["app_id_str"] = df_app_full_info["app_id"].astype(str)

In [34]:
# Convert app id list to strings
list_app_ids_ios_str = [str(a) for a in list_app_ids_ios]
list_app_ids_android_str = [str(a) for a in list_app_ids_android]

In [36]:
list_app_ids_str = list_app_ids_ios_str + list_app_ids_android

In [35]:
# Keep only rows for those app IDs
df_app_ios_info = df_app_full_info[
    df_app_full_info["app_id_str"].isin(list_app_ids_ios_str)
]
df_app_android_info = df_app_full_info[
    df_app_full_info["app_id_str"].isin(list_app_ids_android_str)
]

In [39]:
# Sort by app and date (equivalent to your per-app sorted(..., key=lambda x["d"]))
df_perf = df_perf.sort_values(["aid_str", "d"])

In [ ]:
# For each app, get a list of dicts (one dict per performance row)
perf_lists = (
    df_perf
    .groupby("aid_str", sort=False)
    .apply(lambda g: g.to_dict("records"))
)